# Reconnaissance vocale (ASR) en éwé — Réentraînement continu de MMS (v2, + WaxalNLP)

> **Objectif de ce notebook (v2).** Le modèle a déjà été entraîné une première fois
> (publié sous `romaricnadjire/mms-ewe-asr`, **WER test ≈ 13.5 %**). Ici on **poursuit**
> cet entraînement sur de **nouvelles données WaxalNLP** (combinées au jeu d'origine),
> sans repartir de zéro, et on enregistre le résultat dans un **nouveau dépôt**
> `romaricnadjire/mms-ewe-asr-v2` (l'ancien modèle reste intact).

**ASR** = *Automatic Speech Recognition* : transformer un **son** (la voix) en **texte**.
Contrairement à Whisper, on utilise ici **MMS** (*Massively Multilingual Speech*) de Meta,
qui prend en charge l'**éwé nativement** (parmi >1000 langues). C'est souvent **meilleur**
que Whisper pour les langues africaines peu dotées.

## MMS, comment ça marche ? (CTC vs encodeur-décodeur)

MMS repose sur **wav2vec2** et la perte **CTC** (*Connectionist Temporal Classification*).
La grande différence avec Whisper :

| | Whisper | MMS (wav2vec2-CTC) |
|---|---|---|
| Architecture | encodeur → **décodeur** (génératif) | encodeur **seul** + tête CTC |
| Entrée | spectrogramme log-Mel | **forme d'onde brute** (16 kHz) |
| Sortie | tokens de sous-mots générés | **un caractère par pas de temps**, fusionnés par CTC |
| Vocabulaire | fixe (BPE multilingue) | **construit à partir de nos transcriptions** |

**CTC en une phrase :** le modèle prédit, pour chaque petite tranche de temps, une lettre
(ou un « blanc »). CTC se charge ensuite de fusionner les répétitions et de retirer les blancs
pour reconstruire le texte — sans avoir besoin d'aligner manuellement audio et caractères.

```mermaid
flowchart LR
    A[Audio .flac 48 kHz] -->|reechantillonnage 16 kHz| B[Forme d onde brute]
    B --> C[Encodeur wav2vec2 1B]
    C --> D[Adaptateur de langue eve]
    D --> E[Tete CTC: 1 lettre par pas de temps]
    E -->|fusion CTC| F[Texte eve]
```

## L'astuce MMS : les **adaptateurs** de langue

MMS-1B contient **1 milliard** de paramètres partagés + de petits **adaptateurs** (~2 M
paramètres) spécifiques à chaque langue. On **gèle le gros modèle** et on n'entraîne que
l'adaptateur éwé : très rapide, peu de VRAM. **Ici on ne réinitialise pas** l'adaptateur
(`init_adapter_layers()`) : on **recharge celui déjà affiné** (`romaricnadjire/mms-ewe-asr`)
et on le **poursuit** sur les nouvelles données, avec un **taux d'apprentissage réduit**.


## 0. Installation des dépendances

- `transformers`, `datasets` : modèle + données
- `evaluate`, `jiwer` : métriques **WER** / **CER**
- `librosa`, `soundfile` : lecture/rééchantillonnage audio
- `accelerate` : entraînement optimisé

In [ ]:
!pip install -q transformers datasets evaluate jiwer accelerate librosa soundfile

## 1. Imports et configuration

On force **un seul GPU** (Kaggle T4 x2) et on réduit la fragmentation mémoire **avant**
d'importer `torch`. `TARGET_LANG = "ewe"` est le code ISO 639-3 de l'éwé reconnu par MMS.

Nouveauté v2 : `PREV_MODEL_REPO` pointe vers le **modèle déjà entraîné** (point de départ) et
on écrit les résultats dans un **nouveau dossier / dépôt** (`OUTPUT_DIR`, `NEW_HUB_REPO`) afin
de **ne pas écraser** le run précédent.

In [ ]:
import os
import re
import json

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---- Caches sur le disque scratch (évite la saturation de / et /tmp sur Kaggle) ----
# Sur Kaggle, le conteneur ('/' et '/tmp') est petit ; '/kaggle/temp' est sur le grand disque.
_SCRATCH = "/kaggle/temp" if os.path.isdir("/kaggle") else os.path.join(os.getcwd(), ".cache")
for _var, _sub in [
    ("HF_HOME", "huggingface"),
    ("HF_DATASETS_CACHE", "huggingface/datasets"),
    ("HF_HUB_CACHE", "huggingface/hub"),
    ("TRANSFORMERS_CACHE", "huggingface/transformers"),
    ("TMPDIR", "tmp"),
]:
    _p = os.path.join(_SCRATCH, _sub)
    os.makedirs(_p, exist_ok=True)
    os.environ[_var] = _p

from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Union

import torch
import numpy as np
import evaluate
from datasets import load_dataset, Audio, DatasetDict, concatenate_datasets
from transformers import (
    AutoProcessor,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Configuration ----
MODEL_NAME      = "facebook/mms-1b-all"             # modèle de base d'origine (référence)
PREV_MODEL_REPO = "romaricnadjire/mms-ewe-asr"     # <-- point de départ : l'adaptateur DÉJÀ entraîné
TARGET_LANG     = "ewe"                             # code ISO 639-3 de l'eve (adaptateur MMS)
OUTPUT_DIR      = "./output/mms-ewe-v2"            # nouveau dossier (ne pas écraser le run précédent)
NEW_HUB_REPO    = "romaricnadjire/mms-ewe-asr-v2"  # nouveau dépôt (l'ancien est préservé)

DATASET_ID      = "romaricnadjire/ewe-asr-whisper"  # jeu d'origine (déjà utilisé au 1er run)
WAXAL_SOURCE    = "google/WaxalNLP"                # nouvelles données (config "ewe_asr")
COMBINE_ORIGINAL = True                            # True = WaxalNLP + jeu d'origine (recommandé)

SAMPLING_RATE   = 16_000                            # MMS attend du 16 kHz
MAX_AUDIO_SEC   = 20.0                              # filtrer les audios trop longs
MAX_TRAIN_SAMPLES = None                            # plafonner le train combiné (p.ex. 8000) pour un essai
MAX_WAXAL_SAMPLES = None                            # plafonner uniquement WaxalNLP si besoin

LEARNING_RATE   = 1e-4                              # LR RÉDUIT : on poursuit un adaptateur déjà bon
NUM_EPOCHS      = 2                                 # nombre d'époques sur le jeu combiné

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("device :", device, "| langue cible :", TARGET_LANG)
print("Départ :", PREV_MODEL_REPO, "-> sortie :", NEW_HUB_REPO)


## 2. Chargement et fusion des données (jeu d'origine + WaxalNLP)

On combine **deux sources éwé** : le jeu déjà utilisé au 1er run
(`romaricnadjire/ewe-asr-whisper`, colonne `transcription` → `sentence`) et les **nouvelles
données** `google/WaxalNLP` (config `ewe_asr`). On ne garde que `audio` + `sentence`, on
rééchantillonne tout à **16 kHz**, puis on **concatène** les splits `train` et `validation`.

Deux jeux de test restent **séparés** : `test` (origine) — directement **comparable** au run
précédent (WER 13.5 %) — et `test_waxal` (nouveau domaine). Le paramètre `columns=` au chargement
de WaxalNLP contourne une colonne résiduelle (`__index_level_0__`) qui provoque sinon une erreur.

In [ ]:
# ---- Authentification Hugging Face ----
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

def _get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

HF_TOKEN_READ  = _get_secret('HF_TOKEN_READ')  or _get_secret('HF_TOKEN')
HF_TOKEN_WRITE = _get_secret('HF_TOKEN_WRITE') or HF_TOKEN_READ
if HF_TOKEN_READ:
    os.environ['HF_TOKEN'] = HF_TOKEN_READ

# --- 1) Jeu d'origine : snapshot_download bulk (dataset modeste, eprouve) ---
from huggingface_hub import snapshot_download

ORIG_LOCAL_DIR = snapshot_download(
    repo_id    = DATASET_ID,
    repo_type  = 'dataset',
    token      = HF_TOKEN_READ or True,
    max_workers= 4,
    local_dir  = os.path.join(_SCRATCH, 'ewe_asr_orig'),
)
orig = load_dataset('audiofolder', data_dir=ORIG_LOCAL_DIR, token=HF_TOKEN_READ or True)
if 'transcription' in orig['train'].column_names:
    orig = orig.rename_column('transcription', 'sentence')
orig = DatasetDict({
    s: orig[s].remove_columns([c for c in orig[s].column_names if c not in ['audio','sentence']])
    for s in orig
}).cast_column('audio', Audio(sampling_rate=SAMPLING_RATE))
print('Origine :', {s: len(orig[s]) for s in orig})

# --- 2) WaxalNLP en STREAMING ---
# streaming=True : les shards parquet sont telecharges au fur et a mesure,
# pas tous d'un coup (centaines de Go au total -> timeout Kaggle).

def _load_waxal_stream(split):
    """Charge un split WaxalNLP (ewe_asr) en streaming."""
    ds = load_dataset(
        WAXAL_SOURCE, 'ewe_asr',
        split=split,
        streaming=True,
        token=HF_TOKEN_READ or True,
    )
    ds = ds.rename_column('transcription', 'sentence')
    try:
        extra = [c for c in ds.features if c not in {'audio', 'sentence'}]
        if extra:
            ds = ds.remove_columns(extra)
    except Exception:
        pass
    return ds.cast_column('audio', Audio(sampling_rate=SAMPLING_RATE))

def _materialize(stream_ds, max_n=None):
    """Convertit un IterableDataset -> Dataset classique (pour val/test)."""
    from datasets import Dataset as _DS
    rows = []
    for i, row in enumerate(stream_ds):
        if max_n is not None and i >= max_n:
            break
        rows.append({'audio': row['audio'], 'sentence': row['sentence']})
    result = _DS.from_list(rows)
    return result.cast_column('audio', Audio(sampling_rate=SAMPLING_RATE))

# Val et test WaxalNLP : petits (< 300 ex) -> materialiser maintenant
print('Chargement WaxalNLP val + test (materialisation)...')
wax_val  = _materialize(_load_waxal_stream('validation'))
wax_test = _materialize(_load_waxal_stream('test'))
print(f'  WaxalNLP : val={len(wax_val)}  test={len(wax_test)}')

# Train WaxalNLP : IterableDataset (streaming progressif)
wax_train_stream = _load_waxal_stream('train')
if MAX_WAXAL_SAMPLES:
    wax_train_stream = wax_train_stream.take(MAX_WAXAL_SAMPLES)

# --- 3) Fusion ---
if COMBINE_ORIGINAL:
    val_combined = concatenate_datasets([orig['validation'], wax_val])
else:
    val_combined = wax_val

ds = DatasetDict({
    'validation' : val_combined,
    'test'       : orig['test'],       # ORIGINE -> comparable au run precedent
    'test_waxal' : wax_test,           # WaxalNLP -> mesure sur le nouveau domaine
})

# Train : IterableDataset chaine (orig converti + waxal streaming)
if COMBINE_ORIGINAL:
    orig_train_iter = orig['train'].to_iterable_dataset(num_shards=16)
    ds_train_iter   = concatenate_datasets([orig_train_iter, wax_train_stream])
else:
    ds_train_iter = wax_train_stream

print(f'Val combinee : {len(ds["validation"])}  |  Test origine : {len(ds["test"])}')
print('Train : IterableDataset streaming (~15k orig + ~15k waxal)')


In [ ]:
# Inspecter un exemple depuis le jeu d'origine (acces direct par index)
ex = orig['train'][0]
print('Phrase        :', ex['sentence'])
print('Sampling rate :', ex['audio']['sampling_rate'])
print('Duree (s)     :', round(len(ex['audio']['array']) / ex['audio']['sampling_rate'], 2))


## 3. Normalisation du texte

Pour un modèle CTC au niveau **caractère**, on nettoie d'abord la ponctuation (qui ne
s'« entend » pas) et on met en minuscules. **Important :** on **conserve** toutes les
lettres et diacritiques de l'éwé (ɖ, ɔ, ŋ, ɛ, ʋ, ƒ, accents de ton…) — ce sont des sons,
pas de la ponctuation. La mise en minuscule de Python gère correctement ces lettres
(Ɖ→ɖ, Ɔ→ɔ, Ŋ→ŋ, Ɛ→ɛ…).

In [ ]:
# Normalisation appliquee aux splits val/test (Datasets classiques).
# Le train streaming est normalise directement dans prepare_batch (section 6).
chars_to_remove_regex = r'[\,\?\.\!\-\;\:\"\"\"\u201e\u201f\u2018\u2019\u00ab\u00bb\u2026()\[\]/]'

def remove_special_characters(batch):
    text = re.sub(chars_to_remove_regex, '', batch['sentence'])
    batch['sentence'] = text.lower().strip()
    return batch

ds = ds.map(remove_special_characters, desc='Nettoyage du texte (val/test)')
print('Exemple nettoye (val) :', ds['validation'][0]['sentence'])


## 4. Réutilisation du vocabulaire éwé existant

En continuation, on **ne reconstruit pas** le vocabulaire : la tête CTC du modèle déjà entraîné
est alignée sur un vocabulaire précis. On **recharge donc le processeur** du modèle précédent
(`romaricnadjire/mms-ewe-asr`) via `AutoProcessor.from_pretrained(...)`, ce qui garantit que
chaque caractère garde **exactement le même identifiant** qu'au 1er run.

In [ ]:
# On RECHARGE le processeur du modèle précédent (vocabulaire éwé identique au 1er run).
processor = AutoProcessor.from_pretrained(PREV_MODEL_REPO, token=os.environ.get("HF_TOKEN"))
processor.tokenizer.set_target_lang(TARGET_LANG)
processor.save_pretrained(OUTPUT_DIR)
print("Processeur rechargé depuis", PREV_MODEL_REPO, "| taille vocab :", len(processor.tokenizer))


## 5. Contrôle de couverture du vocabulaire (OOV)

Comme on réutilise le vocabulaire figé du 1er run, il faut vérifier que les **nouvelles**
transcriptions (WaxalNLP) n'introduisent pas de caractères absents. On compte les caractères
**hors-vocabulaire** (*OOV*) : ils seront encodés en `[UNK]`. Si le pourcentage est très faible,
l'impact est négligeable et on peut poursuivre sans toucher au vocabulaire.

In [ ]:
# Jeu de caracteres connu (vocabulaire fige) : '|' represente l'espace
known = set(processor.tokenizer.get_vocab().keys())
for tok in ['[UNK]', '[PAD]', '<s>', '</s>']:
    known.discard(tok)
if '|' in known:
    known.discard('|')
    known.add(' ')

# Caracteres presents dans val/test + orig train (Datasets classiques).
# Le train WaxalNLP (streaming) est ignore ici pour eviter un telechargement inutile.
seen = {}
for split in ds:  # validation, test, test_waxal
    for txt in ds[split]['sentence']:
        for ch in txt:
            seen[ch] = seen.get(ch, 0) + 1
for txt in orig['train']['sentence']:
    for ch in txt:
        seen[ch] = seen.get(ch, 0) + 1

oov = sorted(c for c in seen if c not in known)
n_chars = sum(seen.values())
n_oov   = sum(seen[c] for c in oov)
print(f'Caracteres distincts vus : {len(seen)} | hors-vocabulaire : {len(oov)}')
print(f'Occurrences OOV : {n_oov} / {n_chars} ({100*n_oov/max(n_chars,1):.4f} %)')
if oov:
    print('Exemples OOV :', ''.join(oov)[:60])
    print('-> ces caracteres seront encodes en [UNK] (negligeable si le % est tres faible).')
else:
    print('Couverture parfaite : aucun caractere hors-vocabulaire.')
print('(Train WaxalNLP streaming ignore dans ce check)')


## 6. Préparation des données

Pour chaque exemple :
- `input_values` = la **forme d'onde** normalisée (et non un spectrogramme) ;
- `labels` = la phrase encodée en identifiants de caractères.

On applique cette transformation **à la volée** (`set_transform`, en mémoire) plutôt que
`map()` : avec de l'audio, `map()` écrirait sur disque des dizaines de Go et **saturerait
le disque Kaggle**. Les audios trop longs sont **tronqués** à `MAX_AUDIO_SEC` (au lieu d'être
filtrés) pour limiter la VRAM.

In [ ]:
# Deux cas selon le type de dataset :
# - Train (IterableDataset) : .map() lazy -- prepare + telecharge a la volee.
# - Val/Test (Dataset classique) : set_transform, aucune ecriture disque.

_MAX_LEN = int(MAX_AUDIO_SEC * SAMPLING_RATE)
_norm_re  = re.compile(chars_to_remove_regex)

def _normalize_text(text):
    return _norm_re.sub('', text or '').lower().strip()

def prepare_batch(batch):
    out = {'input_values': [], 'labels': []}
    for audio, sentence in zip(batch['audio'], batch['sentence']):
        sentence = _normalize_text(sentence)   # idempotent (no-op si deja normalise)
        iv = processor(audio['array'], sampling_rate=audio['sampling_rate']).input_values[0]
        out['input_values'].append(iv[:_MAX_LEN])
        out['labels'].append(processor(text=sentence).input_ids)
    return out

# Train (streaming) : map lazy
ds_train_prep = ds_train_iter.map(
    prepare_batch,
    batched=True,
    remove_columns=['audio', 'sentence'],
)

# Val / Test (Datasets classiques) : set_transform a la volee
ds_eval = ds   # DatasetDict : validation, test, test_waxal
for _split in ds_eval:
    ds_eval[_split].set_transform(prepare_batch)

print('ds_train_prep : IterableDataset (streaming + map)')
print(f'ds_eval       : {list(ds_eval.keys())} (set_transform)')


## 7. Le *data collator* CTC (padding dynamique)

Formes d'onde et transcriptions ont des longueurs variables. Le collator complète
(*padding*) séparément :
- les `input_values` (audio) via le `feature_extractor` ;
- les `labels` via le `tokenizer`, en mettant les positions de padding à **-100** pour
  qu'elles soient **ignorées** par la perte CTC.

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        # padding de l'audio
        input_features = [{"input_values": f["input_values"]} for f in features]
        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        # padding des labels
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, padding=self.padding, return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)


## 8. Métriques : WER et CER

Pour un modèle CTC, la prédiction est l'**argmax** des logits à chaque pas de temps ;
`batch_decode` applique la fusion CTC (suppression des répétitions et des blancs).
Le **WER** (mots) et le **CER** (caractères) mesurent le taux d'erreur — **plus bas = mieux**.

In [ ]:
metric_wer = evaluate.load("wer")
metric_cer = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    # remettre le pad_token a la place des -100 pour decoder les references
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    wer = 100 * metric_wer.compute(predictions=pred_str, references=label_str)
    cer = 100 * metric_cer.compute(predictions=pred_str, references=label_str)
    return {"wer": wer, "cer": cer}


## 9. Rechargement du modèle déjà entraîné (continuation)

C'est **la** différence clé avec le 1er run :

1. on recharge directement le modèle fine-tuné `romaricnadjire/mms-ewe-asr` (adaptateur éwé
   **déjà appris** + tête CTC alignée sur le vocabulaire existant) ;
2. on **n'appelle PAS** `init_adapter_layers()` — sinon on remettrait l'adaptateur à zéro et on
   perdrait tout l'entraînement précédent ;
3. `freeze_base_model()` **gèle le milliard de paramètres** partagés ;
4. on **réactive uniquement** les poids de l'adaptateur → on poursuit l'apprentissage de ~2 M
   paramètres, avec un **LR réduit**.

Résultat : on repart du modèle performant et on l'améliore, au lieu de tout réapprendre.

In [ ]:
# On RECHARGE le modèle déjà fine-tuné (incl. son adaptateur éwé et sa tête CTC).
model = Wav2Vec2ForCTC.from_pretrained(
    PREV_MODEL_REPO,
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    layerdrop=0.0,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    token=os.environ.get("HF_TOKEN"),
)

# IMPORTANT (continuation) : PAS de model.init_adapter_layers() ici,
# sinon on remettrait l'adaptateur à zéro et on perdrait l'entraînement précédent.
model.freeze_base_model()                       # on gèle le gros modèle partagé
for param in model._get_adapters().values():    # on ne ré-entraîne QUE l'adaptateur éwé
    param.requires_grad = True

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"Parametres entraines : {n_train:,} / {n_total:,} ({100*n_train/n_total:.3f} %)")


## 10. Entraînement (poursuite avec un LR réduit)

On utilise un `Trainer` **standard** (pas `Seq2SeqTrainer` : CTC ne « génère » pas, il
classe chaque pas de temps). Comme on **poursuit** un adaptateur déjà bon, on baisse le
**taux d'apprentissage** (`LEARNING_RATE = 1e-4`, contre `1e-3` au 1er run) et on raisonne en
**époques** sur le jeu combiné plutôt qu'en `max_steps`. Reprise automatique sur le dernier
checkpoint pour les runs Kaggle en arrière-plan.

In [ ]:
# Avec un IterableDataset, num_train_epochs est ignore (pas de len()).
# max_steps remplace : ~30k ex, batch effectif=8 -> ~3750 steps/epoque -> 7500 pour 2.
MAX_STEPS = 7_500

training_args = TrainingArguments(
    output_dir = OUTPUT_DIR,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 2,        # batch effectif = 8
    per_device_eval_batch_size  = 4,
    learning_rate = LEARNING_RATE,
    warmup_steps  = 200,
    max_steps     = MAX_STEPS,              # remplace num_train_epochs (IterableDataset)
    gradient_checkpointing = True,
    fp16 = (device == 'cuda'),
    eval_strategy = 'steps',
    eval_steps    = 500,
    save_steps    = 500,
    logging_steps = 25,
    load_best_model_at_end = True,
    metric_for_best_model  = 'wer',
    greater_is_better      = False,
    save_total_limit       = 2,             # garder 2 checkpoints locaux
    dataloader_num_workers = 2,
    disable_tqdm           = True,
    report_to              = 'none',
    # --- Resilience Kaggle : push automatique de chaque checkpoint sur HF Hub ---
    push_to_hub      = True,
    hub_model_id     = NEW_HUB_REPO,
    hub_token        = HF_TOKEN_WRITE,
    hub_strategy     = 'checkpoint',        # pousse a chaque save_steps=500
    hub_private_repo = True,
)

trainer = Trainer(
    model            = model,
    args             = training_args,
    train_dataset    = ds_train_prep,           # IterableDataset
    eval_dataset     = ds_eval['validation'],   # Dataset classique
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
    processing_class = processor,
)
print(f'Trainer MMS pret. max_steps={MAX_STEPS}')


In [ ]:
# Sanity check + baseline : le modèle rechargé doit reproduire la perf publiée
# (~13.5 % WER / ~2.1 % CER) sur le TEST D'ORIGINE, avant toute poursuite d'entraînement.
baseline_metrics = trainer.evaluate(ds_eval["test"], metric_key_prefix="baseline")

baseline_summary = {
    "split": "test_original",
    "wer": float(baseline_metrics.get("baseline_wer")),
    "cer": float(baseline_metrics.get("baseline_cer")),
    "loss": float(baseline_metrics.get("baseline_loss")) if baseline_metrics.get("baseline_loss") is not None else None,
}

baseline_path = Path(OUTPUT_DIR) / "baseline_metrics.json"
with open(baseline_path, "w", encoding="utf-8") as f:
    json.dump(baseline_summary, f, ensure_ascii=False, indent=2)

print("=== BASELINE = modèle précédent rechargé (test d'origine) ===")
print(f"WER : {baseline_summary['wer']:.2f} %   (référence publiée : 13.52 %)")
print(f"CER : {baseline_summary['cer']:.2f} %   (référence publiée : 2.11 %)")
print(f"Sauvegardé dans : {baseline_path}")
if baseline_summary["wer"] > 25:
    print("⚠️  WER anormalement élevé : vérifier le chargement du modèle/processeur (vocab).")

In [ ]:
from huggingface_hub import HfApi, snapshot_download as _snap_dl

last_ckpt = None
output_path = Path(OUTPUT_DIR)

# 1. Chercher un checkpoint LOCAL (priorite maximale)
if output_path.is_dir():
    ckpts = sorted(
        [d for d in output_path.iterdir()
         if d.is_dir() and d.name.startswith('checkpoint-')],
        key=lambda d: int(d.name.split('-')[-1]),
    )
    if ckpts:
        last_ckpt = str(ckpts[-1])
        print(f'Reprise locale : {last_ckpt}')

# 2. Sinon, telecharger le dernier checkpoint depuis le HUB
if last_ckpt is None and HF_TOKEN_WRITE:
    try:
        api = HfApi()
        entries = list(api.list_repo_tree(
            NEW_HUB_REPO, repo_type='model',
            token=HF_TOKEN_WRITE, recursive=False,
        ))
        ckpt_dirs = sorted(
            {e.rfilename.split('/')[0] for e in entries
             if e.rfilename.startswith('checkpoint-')},
            key=lambda x: int(x.split('-')[-1]),
        )
        if ckpt_dirs:
            latest = ckpt_dirs[-1]
            print(f'Telechargement checkpoint Hub : {NEW_HUB_REPO}/{latest} ...')
            _snap_dl(
                repo_id        = NEW_HUB_REPO,
                repo_type      = 'model',
                token          = HF_TOKEN_WRITE,
                allow_patterns = [f'{latest}/*'],
                local_dir      = OUTPUT_DIR,
            )
            last_ckpt = str(output_path / latest)
            print(f'Reprise Hub : {last_ckpt}')
        else:
            print('Aucun checkpoint sur le Hub -> entrainement from scratch.')
    except Exception as e:
        print(f'Hub inaccessible ({e}) -> entrainement from scratch.')

if last_ckpt is None:
    print('Aucun checkpoint -> entrainement from scratch.')

train_result = trainer.train(resume_from_checkpoint=last_ckpt)

# Sauvegarde finale locale
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
model.save_pretrained(OUTPUT_DIR)
print(f'\nModele MMS sauvegarde : {OUTPUT_DIR}')
print(f'Loss train finale : {train_result.training_loss:.4f}')


## 11. Évaluation finale + transcription d'un exemple

In [ ]:
# Évaluation après poursuite d'entraînement
final_val   = trainer.evaluate(ds_eval["validation"],  metric_key_prefix="final_val")
final_test  = trainer.evaluate(ds_eval["test"],        metric_key_prefix="final_test")
final_waxal = trainer.evaluate(ds_eval["test_waxal"],  metric_key_prefix="final_waxal")

baseline_path = Path(OUTPUT_DIR) / "baseline_metrics.json"
baseline_summary = None
if baseline_path.exists():
    with open(baseline_path, "r", encoding="utf-8") as f:
        baseline_summary = json.load(f)

print("=== APRÈS POURSUITE D'ENTRAÎNEMENT ===")
print(f"Validation combinée  WER : {final_val['final_val_wer']:.2f} %  | CER : {final_val['final_val_cer']:.2f} %")
print(f"Test ORIGINE         WER : {final_test['final_test_wer']:.2f} %  | CER : {final_test['final_test_cer']:.2f} %")
print(f"Test WaxalNLP        WER : {final_waxal['final_waxal_wer']:.2f} %  | CER : {final_waxal['final_waxal_cer']:.2f} %")

report = {
    "previous_published": {"split": "test_original", "wer": 13.52, "cer": 2.11},
    "baseline_reloaded_test_original": baseline_summary,
    "final_validation_combined": {
        "wer": float(final_val["final_val_wer"]),
        "cer": float(final_val["final_val_cer"]),
        "loss": float(final_val["final_val_loss"]) if final_val.get("final_val_loss") is not None else None,
    },
    "final_test_original": {
        "wer": float(final_test["final_test_wer"]),
        "cer": float(final_test["final_test_cer"]),
        "loss": float(final_test["final_test_loss"]) if final_test.get("final_test_loss") is not None else None,
    },
    "final_test_waxal": {
        "wer": float(final_waxal["final_waxal_wer"]),
        "cer": float(final_waxal["final_waxal_cer"]),
        "loss": float(final_waxal["final_waxal_loss"]) if final_waxal.get("final_waxal_loss") is not None else None,
    },
}

# Gain par rapport au modèle précédent, mesuré sur le MÊME test d'origine
ref_wer = baseline_summary["wer"] if baseline_summary else 13.52
ref_cer = baseline_summary["cer"] if baseline_summary else 2.11
delta_wer = report["final_test_original"]["wer"] - ref_wer
delta_cer = report["final_test_original"]["cer"] - ref_cer
report["delta_test_original"] = {"wer": float(delta_wer), "cer": float(delta_cer)}

print("=== COMPARAISON sur le test d'ORIGINE (vs modèle précédent) ===")
print(f"Delta WER : {delta_wer:+.2f} %   (négatif = amélioration)")
print(f"Delta CER : {delta_cer:+.2f} %")

metrics_path = Path(OUTPUT_DIR) / "asr_mms_v2_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f"Rapport complet sauvegardé dans : {metrics_path}")

In [ ]:
# Transcrire un exemple audio brut du test set
model.eval().to(device)

def transcrire(audio_array, sr=SAMPLING_RATE):
    inputs = processor(audio_array, sampling_rate=sr, return_tensors="pt")
    with torch.no_grad():
        logits = model(inputs.input_values.to(device)).logits
    pred_ids = torch.argmax(logits, dim=-1)
    return processor.batch_decode(pred_ids)[0]

ex = orig["test"][0]
pred = transcrire(ex["audio"]["array"], ex["audio"]["sampling_rate"])
print("Reference :", ex["sentence"])
print("Predit    :", pred)


## 12. Publication du modèle amélioré (nouveau dépôt)

On publie le modèle poursuivi dans un **nouveau dépôt** `romaricnadjire/mms-ewe-asr-v2` :
l'ancien (`romaricnadjire/mms-ewe-asr`) **reste intact** pour comparaison. La publication est
protégée par un garde-fou `DO_PUSH` (mettre à `True` pour publier).

- **Réutiliser à l'inférence :** `Wav2Vec2ForCTC.from_pretrained(NEW_HUB_REPO, target_lang="ewe")`
  puis `processor.tokenizer.set_target_lang("ewe")`.
- **Comparer les deux versions :** le test d'origine est évalué à l'identique → on lit directement
  le gain de WER dans `asr_mms_v2_metrics.json` (`delta_test_original`).


In [ ]:
# Publication du modèle amélioré dans un NOUVEAU dépôt (l'ancien reste intact).
DO_PUSH = False   # passer à True pour publier sur le Hub

if DO_PUSH:
    from huggingface_hub import login
    if not HF_TOKEN_WRITE:
        raise RuntimeError("HF_TOKEN_WRITE manquant (Kaggle Secrets ou .env).")
    login(token=HF_TOKEN_WRITE)
    commit = f"Continuation v2 (WaxalNLP + origine) - depuis {PREV_MODEL_REPO}"
    model.push_to_hub(NEW_HUB_REPO, private=True, commit_message=commit)
    processor.push_to_hub(NEW_HUB_REPO, private=True, commit_message=commit)
    print("Publie :", NEW_HUB_REPO)
else:
    print("Publication ignoree (DO_PUSH = False). Modele sauvegarde localement dans", OUTPUT_DIR)